# 장르별 평균 긍정률 분석

`data/preprocessed/steam_indie_games.csv`를 사용해 Steam 인디게임의 장르별 평균 긍정률을 비교합니다.

**분석 질문**
- 어떤 장르의 평균 긍정률이 높은가?
- 장르별 게임 수와 리뷰 수를 함께 고려했을 때 해석이 달라지는가?
- 출시 전 장르 선택이나 포지셔닝에 참고할 만한 장르는 무엇인가?


In [ ]:
import ast
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display

print('라이브러리 로드 완료')

In [20]:
DATA_PATH = Path('../../../data/preprocessed/steam_indie_games.csv')

df_games = pd.read_csv(DATA_PATH)

print(f'데이터 로드 완료: {df_games.shape[0]:,}개 게임, {df_games.shape[1]:,}개 컬럼')
display(df_games[['appid', 'name', 'positive', 'negative', 'total_reviews', 'genres']].head())


데이터 로드 완료: 9,169개 게임, 20개 컬럼


,appid,name,positive,negative,total_reviews,genres
0,226620,Desktop Dungeons,1912,364,2276,"['Adventure', 'Casual', 'Indie', 'RPG', 'Strat..."
1,230210,ASYLUM,303,45,348,"['Adventure', 'Indie']"
2,251570,7 Days to Die,327889,42157,370046,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul..."
3,252190,Defender's Quest 2: Mists of Ruin,157,98,255,"['Indie', 'RPG', 'Strategy']"
4,269770,Secrets of Grindea,7388,882,8270,"['Action', 'Adventure', 'Indie', 'RPG']"


**해석:** 이 데이터는 게임 단위로 정리되어 있으며, `positive`, `negative`, `total_reviews`, `genres`를 이용해 장르별 긍정률을 계산할 수 있습니다. `genres`는 문자열 형태의 리스트이므로 분석 전에 실제 리스트로 변환한 뒤 한 게임이 여러 장르에 포함될 수 있도록 펼쳐야 합니다.


In [21]:
def parse_genres(value: str) -> list[str]:
    """문자열로 저장된 장르 리스트를 Python list로 변환합니다."""
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(item).strip() for item in parsed if str(item).strip()]
    except (ValueError, SyntaxError):
        pass
    return [item.strip() for item in str(value).split(',') if item.strip()]


df_clean = df_games.copy()
for col in ['positive', 'negative', 'total_reviews']:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

df_clean = df_clean.dropna(subset=['positive', 'negative', 'total_reviews'])
df_clean = df_clean[df_clean['total_reviews'] > 0].copy()
df_clean['positive_rate'] = df_clean['positive'] / df_clean['total_reviews'] * 100
df_clean['genre_list'] = df_clean['genres'].apply(parse_genres)
df_clean = df_clean[df_clean['genre_list'].map(len) > 0].copy()

print(f'분석 가능 게임 수: {len(df_clean):,}개')
print(f"긍정률 범위: {df_clean['positive_rate'].min():.1f}% ~ {df_clean['positive_rate'].max():.1f}%")
display(df_clean[['name', 'total_reviews', 'positive_rate', 'genre_list']].head())


분석 가능 게임 수: 9,169개
긍정률 범위: 0.0% ~ 100.0%


,name,total_reviews,positive_rate,genre_list
0,Desktop Dungeons,2276,84.007030,"[Adventure, Casual, Indie, RPG, Strategy]"
1,ASYLUM,348,87.068966,"[Adventure, Indie]"
2,7 Days to Die,370046,88.607633,"[Action, Adventure, Indie, RPG, Simulation, St..."
3,Defender's Quest 2: Mists of Ruin,255,61.568627,"[Indie, RPG, Strategy]"
4,Secrets of Grindea,8270,89.334946,"[Action, Adventure, Indie, RPG]"


In [22]:
df_genre = df_clean.explode('genre_list').rename(columns={'genre_list': 'genre'})
df_genre['genre'] = df_genre['genre'].astype(str).str.strip()
df_genre = df_genre[df_genre['genre'] != ''].copy()
df_genre = df_genre[df_genre['genre'] != 'Indie'].copy()

print(f'장르 확장 후 행 수: {len(df_genre):,}건 (Indie 제외)')
print(f'고유 장르 수: {df_genre["genre"].nunique():,}개')
display(df_genre[['appid', 'name', 'genre', 'positive_rate', 'total_reviews']].head(10))

장르 확장 후 행 수: 19,955건 (Indie 제외)
고유 장르 수: 8개


,appid,name,genre,positive_rate,total_reviews
0,226620,Desktop Dungeons,Adventure,84.007030,2276
0,226620,Desktop Dungeons,Casual,84.007030,2276
0,226620,Desktop Dungeons,RPG,84.007030,2276
0,226620,Desktop Dungeons,Strategy,84.007030,2276
1,230210,ASYLUM,Adventure,87.068966,348
2,251570,7 Days to Die,Action,88.607633,370046
2,251570,7 Days to Die,Adventure,88.607633,370046
2,251570,7 Days to Die,RPG,88.607633,370046
2,251570,7 Days to Die,Simulation,88.607633,370046
2,251570,7 Days to Die,Strategy,88.607633,370046


**해석:** 한 게임이 여러 장르를 가질 수 있으므로 `explode`를 사용해 게임-장르 단위로 변환했습니다. 예를 들어 한 게임이 `Action`, `Adventure`, `Indie`를 모두 가진다면 세 장르의 집계에 각각 포함됩니다.


In [23]:
MIN_GAMES_PER_GENRE = 30

genre_stats = (
    df_genre
    .groupby('genre')
    .agg(
        game_count=('appid', 'nunique'),
        avg_positive_rate=('positive_rate', 'mean'),
        median_positive_rate=('positive_rate', 'median'),
        total_positive=('positive', 'sum'),
        total_reviews=('total_reviews', 'sum'),
        median_reviews=('total_reviews', 'median'),
    )
    .reset_index()
)

genre_stats['weighted_positive_rate'] = (
    genre_stats['total_positive'] / genre_stats['total_reviews'] * 100
)

genre_stats_filtered = (
    genre_stats[genre_stats['game_count'] >= MIN_GAMES_PER_GENRE]
    .sort_values('avg_positive_rate', ascending=False)
    .reset_index(drop=True)
)

print(f'게임 수 {MIN_GAMES_PER_GENRE}개 이상 장르: {len(genre_stats_filtered):,}개')
display(
    genre_stats_filtered[[
        'genre', 'game_count', 'avg_positive_rate', 'median_positive_rate',
        'weighted_positive_rate', 'median_reviews'
    ]].round(2)
)


게임 수 30개 이상 장르: 8개


,genre,game_count,avg_positive_rate,median_positive_rate,weighted_positive_rate,median_reviews
0,Casual,4051,85.19,89.79,89.57,35.0
1,Action,3995,83.69,87.88,86.61,38.0
2,Adventure,4732,83.59,87.50,87.39,44.0
3,Strategy,1980,83.02,86.36,88.48,52.0
4,Racing,284,82.97,87.60,91.54,27.5
5,Sports,329,82.84,85.71,87.47,37.0
6,RPG,2155,82.42,85.76,84.90,70.0
7,Simulation,2429,79.78,83.87,87.28,56.0


**해석:** `avg_positive_rate`는 장르에 속한 게임들의 긍정률을 단순 평균낸 값입니다. `weighted_positive_rate`는 해당 장르의 긍정 리뷰 합계를 전체 리뷰 합계로 나눈 값이라, 리뷰가 많은 대형 게임의 영향이 더 크게 반영됩니다. 두 지표를 함께 보면 “전반적으로 만족도가 높은 장르”와 “대형 히트작이 끌어올린 장르”를 구분할 수 있습니다.


In [ ]:
plot_df = genre_stats_filtered.sort_values('avg_positive_rate')

fig = px.bar(
    plot_df,
    x='avg_positive_rate',
    y='genre',
    orientation='h',
    color='avg_positive_rate',
    color_continuous_scale='Blues',
    text='avg_positive_rate',
    title=f'장르별 평균 긍정률 (게임 수 {MIN_GAMES_PER_GENRE}개 이상)',
    labels={'avg_positive_rate': '평균 긍정률 (%)', 'genre': '장르'},
    range_x=[0, 100],
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(coloraxis_showscale=False, height=400)
fig.show()

In [ ]:
fig = px.bar(
    plot_df.sort_values('avg_positive_rate', ascending=False),
    x='avg_positive_rate',
    y='genre',
    orientation='h',
    color='avg_positive_rate',
    color_continuous_scale='Reds',
    text='avg_positive_rate',
    title=f'장르별 평균 긍정률 (게임 수 {MIN_GAMES_PER_GENRE}개 이상)',
    labels={'avg_positive_rate': '평균 긍정률 (%)', 'genre': '장르'},
    range_x=[0, 100],
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(coloraxis_showscale=False, height=400)
fig.show()

**해석:** 상위 장르는 해당 장르에 속한 게임들이 전반적으로 좋은 평가를 받는 경향이 있다는 뜻입니다. 다만 이 결과만으로 “그 장르로 만들면 성공한다”고 해석하면 안 됩니다. 장르별 게임 수, 리뷰 수, 가격대, 태그 조합, 플레이 방식까지 함께 봐야 출시 전 전략으로 사용할 수 있습니다.


In [ ]:
fig = px.scatter(
    genre_stats_filtered,
    x='game_count',
    y='avg_positive_rate',
    size='median_reviews',
    color='weighted_positive_rate',
    color_continuous_scale='viridis',
    text='genre',
    size_max=50,
    title='장르별 게임 수와 평균 긍정률',
    labels={
        'game_count': '게임 수',
        'avg_positive_rate': '평균 긍정률 (%)',
        'weighted_positive_rate': '가중 긍정률 (%)',
        'median_reviews': '중위 리뷰 수',
    },
    hover_data={'median_reviews': True, 'weighted_positive_rate': ':.1f'},
)
fig.update_traces(textposition='top center')
fig.update_layout(height=500, yaxis_range=[75, 90])
fig.show()

**해석:** 오른쪽 위에 가까운 장르는 게임 수도 충분하고 평균 긍정률도 높은 장르입니다. 이런 장르는 시장 내 검증 사례가 많으면서도 유저 만족도가 높은 편이므로, 출시 전 벤치마크 후보로 우선 검토할 수 있습니다. 반대로 게임 수는 많지만 평균 긍정률이 낮은 장르는 경쟁이 많거나 유저 기대치가 높은 장르일 수 있습니다.


In [27]:
summary_table = genre_stats_filtered[[
    'genre', 'game_count', 'avg_positive_rate', 'median_positive_rate',
    'weighted_positive_rate', 'median_reviews', 'total_reviews'
]].copy()
summary_table = summary_table.round({
    'avg_positive_rate': 2,
    'median_positive_rate': 2,
    'weighted_positive_rate': 2,
    'median_reviews': 0,
})

print('장르별 평균 긍정률 요약 테이블')
display(summary_table)


장르별 평균 긍정률 요약 테이블


,genre,game_count,avg_positive_rate,median_positive_rate,weighted_positive_rate,median_reviews,total_reviews
0,Casual,4051,85.19,89.79,89.57,35.0,1913194
1,Action,3995,83.69,87.88,86.61,38.0,4184819
2,Adventure,4732,83.59,87.50,87.39,44.0,4527917
3,Strategy,1980,83.02,86.36,88.48,52.0,2179237
4,Racing,284,82.97,87.60,91.54,28.0,170825
5,Sports,329,82.84,85.71,87.47,37.0,87702
6,RPG,2155,82.42,85.76,84.90,70.0,2821889
7,Simulation,2429,79.78,83.87,87.28,56.0,3270806


## 최종 정리

- 장르별 평균 긍정률은 출시 전 시장 탐색에서 “유저 만족도가 비교적 높은 장르”를 찾는 데 유용합니다.
- 단순 평균 긍정률은 장르 내 일반적인 게임의 만족도를 보여줍니다.
- 가중 긍정률은 리뷰가 많은 게임의 영향이 크게 반영되므로, 대형 성공작이 장르 평가를 끌어올렸는지 확인하는 보조 지표로 사용합니다.
- 이 결과는 장르 선택의 단독 근거가 아니라, 가격대·태그·플레이 방식·리뷰 수와 함께 비교해야 합니다.
